In [ ]:
import os,sys
import numpy as np
import scipy.io as scio
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [ ]:
# Parameters
use_existing_model = False
bvp_data = 'BVP/'
ALL_MOTION = [1,2,3,4,5,6]
N_MOTION = len(ALL_MOTION)
T_MAX = 0
n_batch_size = 32

In [ ]:
# Normalize a BVP sample to the range [0, 1]
def normalize_data(data_1):
    data_1_max = np.concatenate((data_1.max(axis=0),data_1.max(axis=1)),axis=0).max(axis=0)
    data_1_min = np.concatenate((data_1.min(axis=0),data_1.min(axis=1)),axis=0).min(axis=0)
    if (len(np.where((data_1_max - data_1_min) == 0)[0]) > 0):
        return data_1
    data_1_max_rep = np.tile(data_1_max,(data_1.shape[0],data_1.shape[1],1))
    data_1_min_rep = np.tile(data_1_min,(data_1.shape[0],data_1.shape[1],1))
    data_1_norm = (data_1 - data_1_min_rep) / (data_1_max_rep - data_1_min_rep)
    return  data_1_norm

# Pad all BVP samples along the time axis to a uniform length T_MAX
def zero_padding(data, T_MAX):
    data_pad = []
    for i in range(len(data)):
        t = np.array(data[i]).shape[2]
        data_pad.append(np.pad(data[i], ((0,0),(0,0),(T_MAX - t,0)), 'constant', constant_values = 0).tolist())
    return np.array(data_pad)

# Convert integer gesture labels to one-hot encoded vectors
def onehot_encoding(label, num_class):
    label = np.array(label).astype('int32')
    label = np.squeeze(label)
    _label = np.eye(num_class)[label-1]
    return _label

# Load 4D array and preprocesses BVP samples
def load_4D_data(path_to_data, motion_sel, specific_folders=None):
    global T_MAX
    data = []
    label = []
    
    for data_root, data_dirs, data_files in os.walk(path_to_data):
        # Check if we should filter by specific folders
        if specific_folders is not None:
            # Get the current folder name from the path
            current_folder = os.path.basename(data_root)
            parent_folder = os.path.basename(os.path.dirname(data_root))
            grandparent_folder = os.path.basename(os.path.dirname(os.path.dirname(data_root)))
            
            # Check if current path contains any of the specific folders
            path_contains_specific = any(folder in data_root for folder in specific_folders)
            
            if not path_contains_specific:
                continue
        
        for data_file_name in data_files:
            file_path = os.path.join(data_root, data_file_name)
            try:
                data_1 = scio.loadmat(file_path)['velocity_spectrum_ro']
                label_1 = int(data_file_name.split('-')[1])
                location = int(data_file_name.split('-')[2])
                orientation = int(data_file_name.split('-')[3])
                repetition = int(data_file_name.split('-')[4])

                # Select Motion
                if (label_1 not in motion_sel):
                    continue
                
                # Normalization
                data_normed_1 = normalize_data(data_1)
                
                # Update T_MAX
                if T_MAX < np.array(data_1).shape[2]:
                    T_MAX = np.array(data_1).shape[2]                
            except Exception:
                continue

            # Save List
            data.append(data_normed_1.tolist())
            label.append(label_1)
    
    # Zero-padding
    data = zero_padding(data, T_MAX)

    # Swap axes
    data = np.swapaxes(np.swapaxes(data, 1, 3), 2, 3) # [N,20,20',T_MAX]=>[N,T_MAX,20,20']
    data = np.expand_dims(data, axis=-1) # [N,T_MAX,20,20]=>[N,T_MAX,20,20,1]

    # Convert label to ndarray [N,T_MAX,20,20,1], label(ndarray): [N,N_MOTION]
    label = np.array(label)
    label = onehot_encoding(label, N_MOTION)

    return data, label

In [ ]:
# Samples from Room 1, user No. 5,10~17
data_train, label_train = load_4D_data(bvp_data, ALL_MOTION, specific_folders=['20181130-VS']) 
print('\nLoaded dataset of ' + str(label_train.shape[0]) + ' samples, each sized ' + str(data_train[0,:,:].shape) + '\n')

In [ ]:
# Samples from Room 1,2,3, user No. 1,3,5,6,7,8,9,10,11,12,13,14,15,16,17
folders_to_load = ['20181130-VS', '20181204-VS', '20181209-VS', '20181211-VS'] 
data_test_1, label_test_1 = load_4D_data(bvp_data, ALL_MOTION, specific_folders=folders_to_load)
print('\nLoaded dataset of ' + str(label_test_1.shape[0]) + ' samples, each sized ' + str(data_test_1[0,:,:].shape) + '\n')

In [ ]:
# Samples from Room 2, user No. 1,6
folders_to_load = ['20181204-VS', '20181209-VS']
data_test_2, label_test_2 = load_4D_data(bvp_data, ALL_MOTION, specific_folders=folders_to_load)
print('\nLoaded dataset of ' + str(label_test_2.shape[0]) + ' samples, each sized ' + str(data_test_2[0,:,:].shape) + '\n')

In [ ]:
# Samples from Room 3, user No.3,7,8,9
data_test_3, label_test_3 = load_4D_data(bvp_data, ALL_MOTION, specific_folders=['20181211-VS'])
print('\nLoaded dataset of ' + str(label_test_3.shape[0]) + ' samples, each sized ' + str(data_test_3[0,:,:].shape) + '\n')

In [ ]:
# Define the data generator
class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, data, labels, batch_size):
        self.data = data
        self.labels = labels
        self.batch_size = batch_size
        self.indexes = np.arange(len(self.data))

    def __len__(self):
        # Compute the number of batches to produce
        return int(np.ceil(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        # Generate indexes of the batch
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]

        # Slice the data and labels
        batch_data = self.data[batch_indexes]
        batch_labels = self.labels[batch_indexes]

        return batch_data, batch_labels

    def on_epoch_end(self):
        # Shuffle indexes after each epoch if needed
        np.random.shuffle(self.indexes)

# Create train, validation dataset and test dataset
test_1_generator = DataGenerator(data_test_1, label_test_1, n_batch_size)
test_2_generator = DataGenerator(data_test_2, label_test_2, n_batch_size)
test_3_generator = DataGenerator(data_test_3, label_test_3, n_batch_size)

# Split the data into training and validation sets
train_data, val_data, train_label, val_label = train_test_split(data_train, label_train, test_size=0.2, shuffle=True)

# Create data generators
training_generator = DataGenerator(train_data, train_label, n_batch_size)
validation_generator = DataGenerator(val_data, val_label, n_batch_size)